# P1–P5: GPU training within a two-hour run budget

Fresh training using the same verified private dataset. No old run or old checkpoint is needed.

**Run:** attach the P1–P5 private dataset, select **GPU T4 ×2**, then **Save Version → Save & Run All**. No settings need changing if the dataset is detected automatically.

This notebook performs **15 evaluation fits + 1 all-five deployment fit**, rather than the previous 75-fit search. Each outer test person has one predeclared validation person drawn from the other four; two compact CNNs compete, then the selected model is refitted on four people. Every test person's recordings stay outside training and model selection.

The notebook uses at most two independent training processes, one per GPU. It has no DataLoader worker processes and no DataParallel. Training samples are gathered directly from GPU memory. Validation runs every second epoch, early stopping is enabled, and one seed is used. This is a faster evaluation protocol with less model-selection coverage and no multi-seed uncertainty estimate.

**Budget:** measured from the first code cell. Data setup has a 10-minute ceiling; evaluation ends by minute 85; all training ends by minute 105; reporting/export has at most 5 minutes, capped by minute 115. The remaining margin is reserved for notebook completion. Background queue time and Kaggle publishing after execution are outside this code's control.

The watchdog can terminate a frozen training process independently of CUDA. If an operation fails or the budget prevents full evaluation, the outputs say **PARTIAL**; a stopped or incomplete run is never presented as a completed five-fold result. The code cannot guarantee a trained model if hardware fails. No actual training runtime or accuracy is claimed until this notebook runs on Kaggle.

In [ ]:
# Configuration — defaults are ready for the existing private P1–P5 dataset.
import time
RUN_STARTED=time.time()
EVALUATION_DEADLINE=RUN_STARTED+85*60
TRAINING_DEADLINE=RUN_STARTED+105*60
FINISH_DEADLINE=RUN_STARTED+115*60

DATA_ROOT=""    # optional explicit folder containing manifest.json and session_log.csv
ZIP_PATH=""     # optional explicit path to the verified ZIP
EXPECTED_ZIP_SHA256="74259c5718d512ffa9cd3bdf63f150badcd6887e0cd2b35f780602e6e13a165c"
STALL_SECONDS=300         # terminate a worker with no phase/checkpoint progress for five minutes
MAX_FOLD_MINUTES=24
MAX_FINAL_MINUTES=18


## Set up the notebook
The notebook contains all required source code. It uses Kaggle's installed packages and does not install or download training code. The raw input is read only. A new output directory is created for each run.

In [ ]:
from pathlib import Path
import os,sys,json,shutil,subprocess,importlib,importlib.util
assert Path("/kaggle/working").is_dir(),"Run on Kaggle; local training is disabled."
for module in ["torch","numpy","pandas","scipy","sklearn","joblib","matplotlib"]:
    assert importlib.util.find_spec(module) is not None,f"Missing Kaggle package: {module}"
import torch
assert torch.cuda.is_available(),"Enable a GPU in Kaggle Settings."
GPUS=min(torch.cuda.device_count(),2)
print("GPUs:",GPUS,"PyTorch:",torch.__version__)
for i in range(GPUS):print(i,torch.cuda.get_device_name(i))
WORK=Path("/kaggle/working")/("FYP_FAST_2H_"+time.strftime("%Y%m%d_%H%M%S",time.gmtime(RUN_STARTED)))
OUT=WORK/"results";OUT.mkdir(parents=True,exist_ok=True)
SOURCE_DIR=WORK/"source";SOURCE_DIR.mkdir(exist_ok=True)


In [ ]:
SOURCES = {'fyp_fast.py': '"""P01-P05 CSI experiments. No training is performed on import.\nResidual baseline inspired by ../Model_Detect-Activity-and-Gestures/cnn.py.\nNew split, quality, normalization, selection, multiscale and checkpoint pipeline.\n"""\nfrom pathlib import Path\nimport os, json, hashlib, random, shutil, zipfile, time, gc, platform\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import butter, sosfilt, sosfilt_zi, welch\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score\nimport joblib\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nVERSION = \'p1p5-fast-2h-v1\'\nPIDS = [\'P01\',\'P02\',\'P03\',\'P04\',\'P05\']\nACTIONS = [\'walk\',\'sitstand\',\'pickup\',\'lightswitch\',\'drink\',\'answerphone\',\'eat\',\'wipetable\',\'tvremote\',\'fall\',\'chop_food\',\'stir_pot\']\nSIGNS = [\'HELLO\',\'THANKYOU\',\'LAMP\',\'FAN\',\'TELEVISION\',\'COOK\',\'WATER\',\'HOT\',\'HUNGRY\',\'TOILET\',\'CALL\',\'HELP\',\'PAIN\',\'ACCIDENT\']\nFINE = ACTIONS + SIGNS\nSTATES = [\'idle\',\'activity\',\'sign\']\nFS = 50\nMAX_WIN = 6*FS\nHOP = 10  # 0.2 seconds\nCANDIDATES = [{\'name\':\'residual4\',\'kind\':\'residual\',\'seconds\':4},\n              {\'name\':\'multiscale4\',\'kind\':\'multiscale\',\'seconds\':4}]\nSESSION_DEADLINE = None\n\nclass SessionBudgetReached(RuntimeError):\n    pass\n\nDEFAULTS = dict(epochs=28, patience=4, min_epochs=8, steps_per_epoch=32,\n                batch_size=256, lr=0.001, weight_decay=0.01, seed=17, tree_estimators=300)\nVALIDATION_PID = {\'P01\':\'P04\',\'P02\':\'P05\',\'P03\':\'P01\',\'P04\':\'P02\',\'P05\':\'P03\'}\n\nPROGRESS_PATH = None\nCURRENT_JOB = \'\'\n\ndef progress(phase):\n    if PROGRESS_PATH is not None:\n        atomic_json(dict(time=time.time(), job=CURRENT_JOB, phase=phase), PROGRESS_PATH)\n\n\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_json(obj,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if hasattr(x,\'item\') else str(x)))\n    os.replace(tmp,path)\n\n\ndef verify_data(root):\n    root=Path(root)\n    manifest=json.loads((root/\'manifest.json\').read_text())\n    assert manifest[\'participants\']==PIDS, \'This notebook requires all five real participants.\'\n    for name,info in manifest[\'files\'].items():\n        f=(root/name).resolve()\n        assert f.is_relative_to(root.resolve()),name\n        assert f.is_file() and f.stat().st_size==info[\'bytes\'],f\'Missing/incomplete: {name}\'\n        assert sha256(f)==info[\'sha256\'],f\'Hash mismatch: {name}\'\n    return sha256(root/\'manifest.json\')\n\n\ndef extract_verified(archive, destination, expected_sha=\'\'):\n    archive=Path(archive); destination=Path(destination)\n    if expected_sha: assert sha256(archive)==expected_sha, \'ZIP SHA256 mismatch: download again.\'\n    with zipfile.ZipFile(archive) as z:\n        assert z.testzip() is None,\'Corrupt ZIP: download again.\'\n        base=destination.resolve()\n        for member in z.infolist():\n            p=(base/member.filename).resolve()\n            assert p.is_relative_to(base),\'Unsafe ZIP path\'\n            assert (member.external_attr>>16)&0o170000 != 0o120000,\'Symlink in ZIP\'\n        z.extractall(destination)\n    root=destination/\'phase1\'\n    verify_data(root)\n    return root\n\n\ndef resolve_csi(root,name):\n    p=Path(root)/Path(name).name\n    if not p.exists(): p=Path(str(p)+\'.gz\')\n    assert p.exists(),p\n    return p\n\n\ndef raw_to_stream(path):\n    # No whole-recording fitted mean, PCA or noise-floor normalization.\n    df=pd.read_csv(path)\n    expected=[\'Packet\',\'Timestamp_us\',\'Channel\']+[q for c in range(52) for q in (f\'SC{c}_Re\',f\'SC{c}_Im\')]\n    assert list(df.columns)==expected,f\'Unexpected CSI schema: {path}\'\n    t=df.Timestamp_us.to_numpy(np.int64)\n    assert len(t)>100 and (np.diff(t)>0).all(),f\'Invalid timestamps: {path}\'\n    assert df.Channel.eq(1).all(),f\'Wrong channel: {path}\'\n    v=df.iloc[:,3:].to_numpy(np.float32)\n    assert np.isfinite(v).all(),f\'Nonfinite CSI: {path}\'\n    a=np.hypot(v[:,::2],v[:,1::2]); x=np.log(np.maximum(a,1e-6))\n    rel=(t-t[0])/1e6\n    grid=np.arange(0,rel[-1],0.01)\n    # Resample at 100 Hz before low-pass and decimation to 50 Hz.\n    xg=np.stack([np.interp(grid,rel,x[:,c]) for c in range(52)],axis=1)\n    idx=np.clip(np.searchsorted(rel,grid),1,len(rel)-1)\n    gap=rel[idx]-rel[idx-1]\n    valid=gap<=0.100\n    severe=gap>0.250\n    sos=butter(2,[0.3,18.0],btype=\'bandpass\',fs=100,output=\'sos\')\n    # Reset filter state following a severe gap. Mark warmup as unavailable.\n    cuts=np.r_[0,np.flatnonzero(severe[:-1] & ~severe[1:])+1,len(grid)]\n    hp=np.zeros_like(xg)\n    for a0,b0 in zip(cuts[:-1],cuts[1:]):\n        if b0<=a0: continue\n        zi=sosfilt_zi(sos)[:,:,None]*xg[a0][None,None,:]\n        hp[a0:b0],_=sosfilt(sos,xg[a0:b0],axis=0,zi=zi)\n        valid[a0:min(a0+300,b0)]=False\n    return dict(t=t[0]+np.round(grid[::2]*1e6).astype(np.int64),\n                x=hp[::2].astype(np.float32), valid=valid[::2],\n                severe=severe[::2], raw_first=int(t[0]),raw_last=int(t[-1]),\n                rows=len(t), raw_rate=len(t)/rel[-1], max_gap_ms=float(np.diff(rel).max()*1000))\n\n\ndef infer_posture(r,part):\n    if isinstance(r.get(\'posture\'),str) and r[\'posture\']: return r[\'posture\'],\'recorded\'\n    # Do not invent original story postures. Explicitly report unknown.\n    if part==\'signs_standing\': return \'standing\',\'part_name\'\n    return \'unknown\',\'unrecorded\'\n\n\ndef prepare(root,work):\n    root=Path(root); work=Path(work); work.mkdir(parents=True,exist_ok=True)\n    fingerprint=verify_data(root)\n    log=pd.read_csv(root/\'session_log.csv\')\n    log=log[log.status.eq(\'kept\') & log.pid.isin(PIDS)].sort_values([\'date\',\'time\']).drop_duplicates([\'pid\',\'part\'],keep=\'last\')\n    assert set(log.pid)==set(PIDS)\n    cache=work/\'cache\'; cache.mkdir(exist_ok=True)\n    streams=[]; audit=[]; events=[]\n    for take in log.itertuples():\n        src=resolve_csi(root,take.csi_file)\n        cp=cache/(src.name.replace(\'.csv\',\'\').replace(\'.gz\',\'\')+\'.npz\')\n        # Source and preprocessing version fingerprint; never trust stale caches.\n        key=hashlib.sha256((fingerprint+VERSION+take.csi_file).encode()).hexdigest()\n        if cp.exists():\n            with np.load(cp,allow_pickle=False) as z:\n                cached={k:z[k] for k in z.files}\n            s=cached if str(cached.pop(\'key\'))==key else raw_to_stream(src)\n        else: s=raw_to_stream(src)\n        np.savez(cp,**s,key=key)\n        s.update(pid=take.pid,part=take.part,file=take.csi_file,date=take.date,rx_gain=float(take.rx_gain))\n        lab=pd.read_csv(root/\'labels\'/Path(take.label_file).name).fillna(\'\')\n        assert (lab.pid==take.pid).all()\n        n=len(s[\'t\']); s[\'state\']=np.full(n,-1,np.int64); s[\'fine\']=np.full(n,-1,np.int64)\n        s[\'event\']=np.full(n,-1,np.int64); s[\'events\']=[]\n        ev=lab[lab.type.isin([\'activity\',\'sign\',\'idle\'])].sort_values(\'t_start_us\')\n        last_end=None\n        for ordinal,(_,r) in enumerate(ev.iterrows()):\n            a,b=int(r.t_start_us),int(r.t_end_us)\n            assert b>a,\'Nonpositive label duration\'\n            assert last_end is None or a>=last_end,\'Overlapping ground-truth labels\'\n            last_end=b\n            posture,posture_source=infer_posture(r,take.part)\n            e=dict(id=f\'{take.csi_file}:{ordinal}\',pid=take.pid,part=take.part,file=take.csi_file,\n                   label=r.label,type=r.type,posture=posture,posture_source=posture_source,\n                   trial=r.trial,script=r.script_id,run=r.run,step=r.step,\n                   context=r.context,intent=r.intent,home_action=r.home_action,\n                   t_start_us=a,t_end_us=b,date=take.date,\n                   i0=int(np.searchsorted(s[\'t\'],a)),i1=int(np.searchsorted(s[\'t\'],b)),\n                   covered=a>=s[\'raw_first\'] and b<=s[\'raw_last\'])\n            if not e[\'covered\']:\n                e.update(eligible=False,reason=\'label_outside_recorded_CSI\',quality_fraction=0.)\n                events.append(e); continue\n            i,j=e[\'i0\'],e[\'i1\']; assert j>i\n            typ=STATES.index(r.type); fine=FINE.index(r.label) if r.label in FINE else -1\n            assert r.type==\'idle\' or fine>=0,f\'Unknown fine label {r.label}\'\n            quality=float(s[\'valid\'][i:j].mean())\n            e.update(eligible=True,reason=\'\',quality_fraction=quality,\n                     severe_gap=bool(s[\'severe\'][i:j].any()))\n            s[\'state\'][i:j]=typ; s[\'fine\'][i:j]=fine; s[\'event\'][i:j]=len(s[\'events\'])\n            s[\'events\'].append(e); events.append(e)\n        # Protocol-defined short inter-trial gaps; no inferred labels in long breaks or stories.\n        if take.part!=\'scripts\':\n            es=s[\'events\']\n            for a,b in zip(es[:-1],es[1:]):\n                i,j=a[\'i1\'],b[\'i0\']\n                if 0<(j-i)/FS<=7:\n                    s[\'state\'][i:j]=1 if take.part in [\'actions\',\'actions_seated\'] else 0\n        s[\'state\'][~s[\'valid\']]=-1; s[\'fine\'][~s[\'valid\']]=-1\n        audit.append(dict(pid=take.pid,part=take.part,rows=int(s[\'rows\']),rate=float(s[\'raw_rate\']),\n                          max_gap_ms=float(s[\'max_gap_ms\']),events=len(s[\'events\']),\n                          usable_fraction=float(s[\'valid\'].mean())))\n        streams.append(s)\n        print(\'Prepared\',take.pid,take.part,round(float(s[\'raw_rate\']),1),\'rows/s\',flush=True)\n    pd.DataFrame(audit).to_csv(work/\'recording_audit.csv\',index=False)\n    pd.DataFrame(events).to_csv(work/\'event_coverage.csv\',index=False)\n    atomic_json(dict(version=VERSION,dataset=fingerprint,fs=FS,classes=FINE,\n                     empty_references=\'QC only, excluded from training to avoid shared-day leakage\'),work/\'dataset_info.json\')\n    return streams,fingerprint\n\n\ndef fit_scale(streams):\n    # Equal stream contribution; only training participants enter this function.\n    stats=[]\n    for s in streams:\n        v=s[\'x\'][s[\'valid\'] & (s[\'state\']>=0)][::5]\n        if len(v): stats.append(np.mean(v.astype(np.float64)**2,axis=0))\n    assert stats\n    return np.maximum(np.sqrt(np.mean(stats,axis=0)),1e-4).astype(np.float32)\n\n\ndef window_ok(s,c):\n    a,b=c-MAX_WIN//2,c+MAX_WIN//2\n    return a>=0 and b<=len(s[\'x\']) and not s[\'severe\'][a:b].any() and s[\'valid\'][a:b].mean()>=0.95\n\n\ndef examples(streams):\n    rows=[]\n    for si,s in enumerate(streams):\n        for c in range(MAX_WIN//2,len(s[\'x\'])-MAX_WIN//2+1,HOP):\n            if not window_ok(s,c): continue\n            a,b=c-FS,c+FS\n            v=s[\'state\'][a:b]; vals=np.bincount(v[v>=0],minlength=3)\n            if vals.max()<0.8*(b-a): continue\n            st=int(vals.argmax()); fv=s[\'fine\'][a:b]\n            fine=-1\n            if st!=0:\n                good=fv[fv>=0]\n                if len(good)>=0.8*(b-a):\n                    counts=np.bincount(good,minlength=len(FINE))\n                    if counts.max()>=0.8*(b-a): fine=int(counts.argmax())\n            rows.append((si,c,st,fine))\n    assert rows,\'No valid training windows\'\n    return np.asarray(rows,np.int64)\n\n\nclass WindowDataset(torch.utils.data.Dataset):\n    def __init__(self,streams,rows,scale,seconds):\n        self.streams,self.rows,self.scale,self.width=streams,rows,scale,int(seconds*FS)\n    def __len__(self): return len(self.rows)\n    def __getitem__(self,i):\n        si,c,st,fi=self.rows[i]; half=self.width//2\n        x=(self.streams[si][\'x\'][c-half:c+half]/self.scale).T.copy()\n        return torch.from_numpy(np.clip(x,-20,20)),int(st),int(fi)\n\n\ndef sampling_weights(streams,rows):\n    # Equal states; then equal fine classes; then equal participants within each class.\n    keys=[(int(st),int(fi),streams[si][\'pid\']) for si,c,st,fi in rows]\n    counts=pd.Series(keys).value_counts().to_dict()\n    classes={st:set(fi for a,fi,p in keys if a==st) for st in range(3)}\n    people={(st,fi):set(p for a,b,p in keys if (a,b)==(st,fi)) for st,fi,p in set(keys)}\n    return torch.tensor([1/(counts[k]*len(classes[k[0]])*len(people[k[:2]])) for k in keys],dtype=torch.double)\n\n\nclass ResBlock(nn.Module):\n    def __init__(self,cin,cout,dilation):\n        super().__init__()\n        self.body=nn.Sequential(nn.Conv1d(cin,cout,7,padding=3*dilation,dilation=dilation),\n            nn.BatchNorm1d(cout),nn.GELU(),nn.Conv1d(cout,cout,7,padding=3*dilation,dilation=dilation),nn.BatchNorm1d(cout))\n        self.skip=nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()\n    def forward(self,x): return F.gelu(self.body(x)+self.skip(x))\n\n\nclass MultiBlock(nn.Module):\n    def __init__(self,cin,cout=96):\n        super().__init__()\n        self.reduce=nn.Conv1d(cin,24,1,bias=False)\n        self.branches=nn.ModuleList([nn.Conv1d(24,24,k,padding=k//2,bias=False) for k in [9,19,39]])\n        self.pool=nn.Sequential(nn.MaxPool1d(3,stride=1,padding=1),nn.Conv1d(cin,24,1,bias=False))\n        self.norm=nn.GroupNorm(8,cout)\n        self.skip=nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()\n        self.se=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Conv1d(cout,16,1),nn.GELU(),nn.Conv1d(16,cout,1),nn.Sigmoid())\n    def forward(self,x):\n        z=self.reduce(x)\n        h=self.norm(torch.cat([b(z) for b in self.branches]+[self.pool(x)],dim=1))\n        return F.gelu(h*self.se(h)+self.skip(x))\n\n\nclass CSIModel(nn.Module):\n    def __init__(self,arch):\n        super().__init__(); self.arch=dict(arch)\n        if arch[\'kind\']==\'residual\':\n            self.body=nn.Sequential(nn.BatchNorm1d(52),nn.Conv1d(52,32,5,padding=2),nn.BatchNorm1d(32),nn.GELU(),\n                ResBlock(32,32,1),nn.MaxPool1d(2),ResBlock(32,48,2),nn.MaxPool1d(2),\n                ResBlock(48,64,4),nn.MaxPool1d(2),ResBlock(64,64,8)); width=64\n        else:\n            self.body=nn.Sequential(MultiBlock(52),nn.AvgPool1d(2),MultiBlock(96),nn.AvgPool1d(2),MultiBlock(96)); width=96\n        self.drop=nn.Dropout(0.3); self.state=nn.Linear(width*2,3); self.fine=nn.Linear(width*2,len(FINE))\n    def forward(self,x):\n        h=self.body(x); h=self.drop(torch.cat([h.mean(-1),h.amax(-1)],dim=1))\n        return self.state(h),self.fine(h)\n\n\ndef augment(x):\n    b,c,t=x.shape\n    scale=torch.empty(b,1,1,device=x.device).uniform_(0.8,1.25)\n    noise=torch.randn_like(x)*torch.empty(b,1,1,device=x.device).uniform_(0,0.08)\n    keep=(torch.rand(b,c,1,device=x.device)>0.05)\n    return (x*scale+noise)*keep\n\n\ndef require_kaggle_gpu():\n    assert Path(\'/kaggle/working\').is_dir(),\'Training is restricted to Kaggle, as requested.\'\n    assert torch.cuda.is_available(),\'Enable a GPU in Kaggle Settings before running training.\'\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef checkpoint_save(payload,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(\'.tmp\')\n    with open(tmp,\'wb\') as f:\n        torch.save(payload,f); f.flush(); os.fsync(f.fileno())\n    if path.exists(): shutil.copy2(path,path.with_suffix(\'.previous.pt\'))\n    os.replace(tmp,path)\n\n\ndef trusted_load(path):\n    # Only use checkpoints generated by this notebook / your own saved outputs.\n    try: return torch.load(path,map_location=\'cpu\',weights_only=False)\n    except Exception:\n        prev=Path(path).with_suffix(\'.previous.pt\')\n        if not prev.exists(): raise\n        print(\'Recovering previous checkpoint:\',prev)\n        return torch.load(prev,map_location=\'cpu\',weights_only=False)\n\n\ndef unwrapped(model): return model.module if isinstance(model,nn.DataParallel) else model\n\n\ndef predict(model,streams,scale,arch,batch=256,device=None):\n    device=device or next(model.parameters()).device\n    model.eval(); result=[]\n    width=int(arch[\'seconds\']*FS); half=width//2\n    for s in streams:\n        centers=np.arange(MAX_WIN//2,len(s[\'x\'])-MAX_WIN//2+1,HOP)\n        valid=np.array([window_ok(s,int(c)) for c in centers])\n        ps=np.full((len(centers),3),np.nan,np.float32)\n        pf=np.full((len(centers),len(FINE)),np.nan,np.float32)\n        good=np.flatnonzero(valid)\n        with torch.inference_mode():\n            for start in range(0,len(good),batch):\n                ix=good[start:start+batch]\n                x=np.stack([(s[\'x\'][c-half:c+half]/scale).T for c in centers[ix]])\n                xt=torch.from_numpy(np.clip(x,-20,20)).to(device,non_blocking=True)\n                with torch.autocast(\'cuda\',dtype=torch.float16,enabled=str(device).startswith(\'cuda\')):\n                    a,b=model(xt)\n                ps[ix]=a.float().softmax(-1).cpu().numpy(); pf[ix]=b.float().softmax(-1).cpu().numpy()\n        result.append(dict(stream=s,centers=centers,ps=ps,pf=pf,valid=valid))\n    return result\n\n\ndef segment_table(preds):\n    rows=[]\n    for p in preds:\n        s=p[\'stream\']; c=p[\'centers\']\n        for e in s[\'events\']:\n            if e[\'type\'] not in [\'activity\',\'sign\']: continue\n            m=(c>=e[\'i0\'])&(c<e[\'i1\'])&p[\'valid\']\n            if m.any():\n                ps=p[\'ps\'][m].mean(0); pf=p[\'pf\'][m].mean(0)\n                start,end=(0,len(ACTIONS)) if e[\'type\']==\'activity\' else (len(ACTIONS),len(FINE))\n                pred=FINE[start+pf[start:end].argmax()]\n                state=int(ps.argmax())\n                # Explicit branch is clearer than relying on true event type for end-to-end identity.\n                if state==1: hier=ACTIONS[pf[:len(ACTIONS)].argmax()]\n                elif state==2: hier=SIGNS[pf[len(ACTIONS):].argmax()]\n                else: hier=\'idle\'\n                action_prob=pf[:len(ACTIONS)]; action_prob=action_prob/(action_prob.sum()+1e-12)\n            else:\n                pred=hier=\'unavailable\'; ps=np.array([np.nan]*3); action_prob=np.zeros(len(ACTIONS))\n            rows.append(dict(**e,pred=pred,cnn_pred=pred,hier_pred=hier,p_sign=float(ps[2]),\n                             pred_state=\'unavailable\' if not m.any() else STATES[int(ps.argmax())],\n                             scored_windows=int(m.sum()),action_prob=action_prob))\n    return pd.DataFrame(rows)\n\n\ndef macro(frame,labels,pred=\'pred\'):\n    if frame.empty: return float(\'nan\')\n    labels=[c for c in labels if c in set(frame.label)]\n    return float(f1_score(frame.label,frame[pred],labels=labels,average=\'macro\',zero_division=0))\n\n\ndef validation_score(preds):\n    d=segment_table(preds)\n    scores=[]\n    for story in [False,True]:\n        for typ,labels in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n            g=d[(d.part.eq(\'scripts\')==story)&d.type.eq(typ)]\n            if len(g): scores.append(macro(g,labels))\n    g=d[d.part.eq(\'scripts\')]\n    if len(g): scores.append(float(f1_score(g.type,g.pred_state,labels=[\'activity\',\'sign\'],average=\'macro\',zero_division=0)))\n    return float(np.mean(scores))\n\n\n\n# -------- Action baseline and held-out selection --------\ndef action_features(streams,scale):\n    feats=[]; ids=[]; labels=[]\n    for s in streams:\n        if s[\'part\'] not in [\'actions\',\'actions_seated\']: continue\n        for e in s[\'events\']:\n            if e[\'type\']!=\'activity\': continue\n            c=(e[\'i0\']+e[\'i1\'])//2\n            if not window_ok(s,c): continue\n            x=(s[\'x\'][c-2*FS:c+2*FS]/scale).T\n            power=x*x; env=power.mean(0).reshape(20,-1).mean(1); env=env/(env.sum()+1e-9)\n            freq,psd=welch(x,fs=FS,nperseg=100,axis=1)\n            bands=np.stack([psd[:,(freq>=a)&(freq<b)].sum(1) for a,b in [(0.3,1),(1,2),(2,4),(4,8),(8,18)]])\n            bands=bands/(bands.sum(0,keepdims=True)+1e-9)\n            f=np.concatenate([np.log(power.mean(1)+1e-9),np.mean(np.abs(np.diff(x,axis=1)),axis=1),\n                np.quantile(x,0.9,axis=1)-np.quantile(x,0.1,axis=1),env,bands.mean(1),bands.std(1)])\n            feats.append(f); ids.append(e[\'id\']); labels.append(e[\'label\'])\n    return np.asarray(feats,np.float32),ids,np.asarray(labels)\n\n\ndef train_tree(train_streams,scale,path):\n    require_kaggle_gpu()  # Also disallow CPU classifier training on the laptop.\n    x,_,y=action_features(train_streams,scale)\n    clf=ExtraTreesClassifier(n_estimators=300,min_samples_leaf=2,max_features=0.7,class_weight=\'balanced\',random_state=17,n_jobs=2)\n    clf.fit(x,y)\n    tmp=Path(path).with_suffix(\'.tmp\'); joblib.dump(clf,tmp); os.replace(tmp,path)\n    return clf\n\n\ndef tree_predictions(clf,streams,scale):\n    x,ids,y=action_features(streams,scale)\n    if not len(x): return {}\n    out=np.zeros((len(x),len(ACTIONS)),np.float32)\n    out[:,[ACTIONS.index(c) for c in clf.classes_]]=clf.predict_proba(x)\n    return dict(zip(ids,out))\n\n\ndef fuse_table(frame,tree_probs,weight):\n    frame=frame.copy()\n    frame[\'tree_pred\']=\'unavailable\'\n    for i,r in frame.iterrows():\n        if r[\'id\'] not in tree_probs: continue\n        pe=tree_probs[r[\'id\']]\n        frame.at[i,\'tree_pred\']=ACTIONS[int(pe.argmax())]\n        if r[\'scored_windows\']:\n            prob=(1-weight)*r.action_prob+weight*pe\n            frame.at[i,\'pred\']=ACTIONS[int(prob.argmax())]\n    return frame\n\n\nDEFAULT_INTENT={\'HELLO\':(\'I4\',\'wake_system\'),\'THANKYOU\':(\'I4\',\'confirm_close\'),\n \'LAMP\':(\'I1\',\'toggle_light\'),\'FAN\':(\'I1\',\'toggle_fan\'),\'TELEVISION\':(\'I1\',\'toggle_tv\'),\n \'COOK\':(\'I1\',\'cooking_mode\'),\'WATER\':(\'I2\',\'bring_water_reminder\'),\'HOT\':(\'I2\',\'cool_room\'),\n \'HUNGRY\':(\'I2\',\'food_request\'),\'TOILET\':(\'I2\',\'toilet_assist\'),\'CALL\':(\'I3\',\'call_caregiver\'),\n \'HELP\':(\'I3\',\'notify_caregiver\'),\'PAIN\':(\'I3\',\'health_alert\'),\'ACCIDENT\':(\'I3\',\'emergency_alert\')}\nANCHOR={\'eat\':\'meal\',\'drink\':\'meal\',\'chop_food\':\'meal\',\'stir_pot\':\'meal\',\n        \'tvremote\':\'tv\',\'answerphone\':\'tv\',\'wipetable\':\'cleaning\',\'pickup\':\'cleaning\',\'fall\':\'after_fall\'}\n\n\ndef intent(sign,context):\n    if sign==\'HOT\' and context==\'meal\': return (\'I3\',\'kitchen_hazard_alert\')\n    if sign==\'HELP\' and context==\'after_fall\': return (\'I3\',\'emergency_alert\')\n    return DEFAULT_INTENT.get(sign,(\'unknown\',\'unknown\'))\n\n\ndef true_runs(mask):\n    edges=np.diff(np.r_[False,mask,False].astype(int))\n    return list(zip(np.flatnonzero(edges==1),np.flatnonzero(edges==-1)))\n\n\ndef commands(p,tau):\n    # Offline centered windows; 1-second trailing probability average.\n    ps=p[\'ps\']; valid=p[\'valid\']; n=len(ps)\n    raw=np.nan_to_num(ps[:,2]); kernel=np.ones(5)\n    den=np.convolve(valid.astype(float),kernel,\'full\')[:n]\n    sm=np.convolve(raw,kernel,\'full\')[:n]/np.maximum(den,1)\n    mask=(sm>=tau)&valid&(den>=5)\n    # Consecutive signs in a story can have no idle gap. Split only when a\n    # different predicted identity persists for at least one second.\n    identities=np.nan_to_num(p[\'pf\'][:,len(ACTIONS):]).argmax(1)\n    intervals=[]\n    for a,b in true_runs(mask):\n        start=a; active=int(identities[a]); pending=None; since=a\n        for j in range(a+1,b):\n            label=int(identities[j])\n            if label==active: pending=None\n            elif label!=pending: pending=label; since=j\n            elif j-since+1>=5:\n                intervals.append((start,since)); start=since; active=label; pending=None\n        intervals.append((start,b))\n    out=[]\n    for a,b in intervals:\n        if (b-a)*HOP/FS<1.: continue\n        prob=np.nanmean(p[\'pf\'][a:b,len(ACTIONS):],axis=0)\n        label=SIGNS[int(prob.argmax())]\n        i0=int(p[\'centers\'][a]-HOP//2); i1=int(p[\'centers\'][b-1]+HOP//2)\n        # Context uses only predicted prior activity frames, never true context.\n        recent=(p[\'centers\']<i0)&(p[\'centers\']>=i0-120*FS)&valid\n        good=np.flatnonzero(recent & (np.nan_to_num(ps[:,1])>=0.6))\n        ctx=\'isolated\'\n        for j in good:\n            action=ACTIONS[int(p[\'pf\'][j,:len(ACTIONS)].argmax())]\n            if action in ANCHOR: ctx=ANCHOR[action]\n        cat,home=intent(label,ctx)\n        out.append(dict(i0=i0,i1=i1,label=label,context=ctx,intent=cat,home_action=home))\n    return out\n\n\ndef event_metrics(preds,tau):\n    totals=dict(n_true=0,n_pred=0,detected=0,identified=0,intent_correct=0,false_on_routine=0,routine_seconds=0.)\n    allcommands=[]\n    for p in preds:\n        s=p[\'stream\']\n        if s[\'part\']!=\'scripts\': continue\n        gt=[e for e in s[\'events\'] if e[\'type\']==\'sign\']\n        cmds=commands(p,tau); totals[\'n_true\']+=len(gt); totals[\'n_pred\']+=len(cmds)\n        # Match detection without using sign identity, one-to-one, highest IoU first.\n        pairs=[]\n        for i,c in enumerate(cmds):\n            for j,e in enumerate(gt):\n                inter=max(0,min(c[\'i1\'],e[\'i1\'])-max(c[\'i0\'],e[\'i0\']))\n                union=max(c[\'i1\'],e[\'i1\'])-min(c[\'i0\'],e[\'i0\'])\n                if inter and inter/union>=0.10: pairs.append((inter/union,i,j))\n        used_c=set(); used_e=set()\n        for _,i,j in sorted(pairs,reverse=True):\n            if i in used_c or j in used_e: continue\n            used_c.add(i); used_e.add(j); totals[\'detected\']+=1\n            correct=cmds[i][\'label\']==gt[j][\'label\']; totals[\'identified\']+=int(correct)\n            totals[\'intent_correct\']+=int(correct and cmds[i][\'intent\']==gt[j][\'intent\'] and cmds[i][\'home_action\']==gt[j][\'home_action\'])\n        # Denominator only measured routine time with model coverage, not all story wall time.\n        covered=np.zeros(len(s[\'state\']),bool)\n        for c,v in zip(p[\'centers\'],p[\'valid\']):\n            if v: covered[max(0,c-HOP//2):min(len(covered),c+HOP//2)]=True\n        routine=(s[\'state\']==1)&covered\n        totals[\'routine_seconds\']+=float(routine.sum()/FS)\n        for i,c in enumerate(cmds):\n            midpoint=(c[\'i0\']+c[\'i1\'])//2\n            if i not in used_c and 0<=midpoint<len(routine) and routine[midpoint]: totals[\'false_on_routine\']+=1\n            allcommands.append(dict(pid=s[\'pid\'],file=s[\'file\'],matched=i in used_c,**c))\n    for key,count in [(\'detection\',totals[\'detected\']),(\'identity\',totals[\'identified\'])]:\n        precision=count/max(totals[\'n_pred\'],1); recall=count/max(totals[\'n_true\'],1)\n        totals[key+\'_precision\']=precision; totals[key+\'_recall\']=recall\n        totals[key+\'_f1\']=2*precision*recall/max(precision+recall,1e-12)\n    totals[\'intent_e2e\']=totals[\'intent_correct\']/max(totals[\'n_true\'],1)\n    totals[\'false_commands_per_routine_hour\']=totals[\'false_on_routine\']/max(totals[\'routine_seconds\']/3600,1e-12)\n    totals[\'tau\']=float(tau)\n    return totals,allcommands\n\n\ndef choose_tau(inner_predictions):\n    rows=[]\n    for tau in np.arange(0.2,0.901,0.05):\n        rr=[event_metrics(p,float(tau))[0] for p in inner_predictions]\n        rows.append(dict(tau=float(tau),identity_f1=float(np.mean([r[\'identity_f1\'] for r in rr])),\n                         detection_f1=float(np.mean([r[\'detection_f1\'] for r in rr]))))\n    # Identity first; detection breaks ties; then prefer a higher threshold.\n    selected=max(rows,key=lambda r:(r[\'identity_f1\'],r[\'detection_f1\'],r[\'tau\']))\n    return selected[\'tau\'],rows\n\n\ndef average_predictions(all_preds):\n    result=[]\n    for pos in range(len(all_preds[0])):\n        p=all_preds[0][pos].copy()\n        # NaNs at invalid frames are intentional; average only valid frames.\n        for key in [\'ps\',\'pf\']:\n            p[key]=np.stack([a[pos][key] for a in all_preds]).mean(0)\n        result.append(p)\n    return result\n\n\ndef save_predictions(preds,path):\n    arrays={}\n    for i,p in enumerate(preds):\n        for key in [\'centers\',\'ps\',\'pf\',\'valid\']: arrays[f\'{i}_{key}\']=p[key]\n        arrays[f\'{i}_file\']=np.array(p[\'stream\'][\'file\'])\n    np.savez_compressed(path,**arrays)\n\n\ndef release(net):\n    net.cpu(); torch.cuda.empty_cache(); gc.collect()\n\n\ndef report_fold(table,preds,tau,folder):\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    table.drop(columns=[\'action_prob\']).to_csv(folder/\'segments.csv\',index=False)\n    metrics=[]; perclass=[]\n    # Common 10 actions /14 signs; expanded results show actual support, never fictitious zeros.\n    slices=[(\'isolated\',table[table.part.ne(\'scripts\')]),(\'stories_oracle_boundaries\',table[table.part.eq(\'scripts\')]),\n            (\'isolated_seated\',table[table.part.isin([\'actions_seated\',\'signs_seated\'])]),\n            (\'isolated_original_parts\',table[table.part.isin([\'actions\',\'signs_standing\'])])]\n    for split,d in slices:\n        for typ,names in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n            g=d[d.type.eq(typ)]\n            if g.empty: continue\n            supported=[n for n in names if n in set(g.label)]\n            values=dict(split=split,type=typ,n=len(g),supported_classes=len(supported),\n                        macro_f1=macro(g,supported),accuracy=float((g.label==g.pred).mean()),\n                        cnn_macro_f1=macro(g,supported,\'cnn_pred\'),\n                        unavailable=int(g.scored_windows.eq(0).sum()))\n            if typ==\'activity\': values[\'common10_macro_f1\']=macro(g[g.label.isin(ACTIONS[:10])],ACTIONS[:10])\n            metrics.append(values)\n            rep=classification_report(g.label,g.pred,labels=supported,output_dict=True,zero_division=0)\n            for label in supported: perclass.append(dict(split=split,type=typ,label=label,**rep[label]))\n    ev,cmd=event_metrics(preds,tau)\n    atomic_json(ev,folder/\'story_events.json\'); pd.DataFrame(cmd).to_csv(folder/\'predicted_commands.csv\',index=False)\n    pd.DataFrame(metrics).to_csv(folder/\'metrics.csv\',index=False)\n    pd.DataFrame(perclass).to_csv(folder/\'per_class.csv\',index=False)\n    st=table[table.part.eq(\'scripts\')]; good=st.p_sign.notna()\n    atomic_json(dict(story_auc=roc_auc_score(st.loc[good,\'type\'].eq(\'sign\'),st.loc[good,\'p_sign\']) if st.loc[good,\'type\'].nunique()==2 else None,\n                     story_hierarchical_accuracy=float(st.label.eq(st.hier_pred).mean()),\n                     story_oracle_boundary_note=\'Fine identity uses known event boundaries/type; event detection and intent metrics do not.\',\n                     missing_prediction_policy=\'Fully covered labels without usable windows remain as unavailable and count as misses.\'),folder/\'context.json\')\n\n\n\n\n\n\n\n\ndef archive_outputs(out,only_fold=None):\n    out=Path(out); base=out.parent\n    name=f\'FYP_{only_fold}_checkpoint_bundle\' if only_fold else \'FYP_all_checkpoints_and_results\'\n    dest=base/(name+\'.zip\'); tmp=dest.with_suffix(\'.zip.part\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n        for p in out.rglob(\'*\'):\n            if not p.is_file() or \'cache\' in p.parts or p.suffix in [\'.tmp\',\'.part\']: continue\n            rel=p.relative_to(out)\n            if only_fold and len(rel.parts)>1 and rel.parts[0] not in [only_fold]: continue\n            if only_fold and len(rel.parts)==1 and rel.name not in [\'fyp_pipeline.py\',\'dataset_info.json\',\'environment.json\',\'recording_audit.csv\',\'event_coverage.csv\']: continue\n            z.write(p,Path(\'results\')/rel)\n    tmp.replace(dest)\n    atomic_json(dict(sha256=sha256(dest),bytes=dest.stat().st_size),dest.with_suffix(\'.zip.json\'))\n    return dest\n\n\n\n\ndef summarize(out):\n    import matplotlib.pyplot as plt\n    out=Path(out); frames=[]; metrics=[]; events=[]\n    for pid in PIDS:\n        if not (out/pid/\'complete.json\').exists(): continue\n        d=pd.read_csv(out/pid/\'segments.csv\'); frames.append(d)\n        m=pd.read_csv(out/pid/\'metrics.csv\'); m[\'pid\']=pid; metrics.append(m)\n        ev=json.loads((out/pid/\'story_events.json\').read_text()); ev[\'pid\']=pid; events.append(ev)\n    if not frames: print(\'No completed folds yet.\'); return\n    d=pd.concat(frames,ignore_index=True); m=pd.concat(metrics,ignore_index=True); ev=pd.DataFrame(events)\n    d.to_csv(out/\'all_out_of_fold_predictions.csv\',index=False); m.to_csv(out/\'fold_metrics.csv\',index=False); ev.to_csv(out/\'story_metrics.csv\',index=False)\n    figures=out/\'figures\'; figures.mkdir(exist_ok=True)\n    fig,ax=plt.subplots(figsize=(10,5))\n    for pid in PIDS:\n        for path in sorted((out/pid/\'models\').glob(\'seed_*/history.csv\')):\n            h=pd.read_csv(path)\n            ax.plot(h.epoch,h.loss,label=f\'{pid} {path.parent.name}\',alpha=0.7)\n    ax.set_title(\'Final fold training losses\'); ax.set_xlabel(\'Epoch\'); ax.set_ylabel(\'Loss\')\n    ax.legend(fontsize=7,ncol=3); fig.tight_layout(); fig.savefig(figures/\'training_curves.png\',dpi=160); plt.show(); plt.close(fig)\n    for typ,names in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n        for source,mask in [(\'isolated\',d.part.ne(\'scripts\')),(\'stories_oracle\',d.part.eq(\'scripts\'))]:\n            g=d[mask & d.type.eq(typ)]\n            if g.empty: continue\n            report=classification_report(g.label,g.pred,labels=names,output_dict=True,zero_division=0)\n            per=pd.DataFrame([dict(label=n,**report[n]) for n in names]); per.to_csv(out/f\'{source}_{typ}_per_class.csv\',index=False)\n            fig,ax=plt.subplots(figsize=(11,5)); values=per[\'f1-score\'].to_numpy()\n            ax.bar(names,values,color=[\'#187b62\' if v>=0.6 else \'#ca8a32\' for v in values]); ax.set_ylim(0,1)\n            ax.set_ylabel(\'F1\'); ax.set_title(f\'{source.replace("_"," ")} — {typ}; pooled held-out predictions\')\n            ax.tick_params(axis=\'x\',rotation=55); fig.tight_layout(); fig.savefig(figures/f\'{source}_{typ}_f1.png\',dpi=160); plt.show(); plt.close(fig)\n            # Rectangular confusion includes idle / unavailable instead of silently dropping missed examples.\n            pred_labels=names+[\'idle\',\'unavailable\']+[n for n in FINE if n not in names]\n            cm=np.array([[(g.label.eq(a)&g.pred.eq(b)).sum() for b in pred_labels] for a in names])\n            used=cm.sum(0)>0; cm=cm[:,used]; cols=np.array(pred_labels)[used]\n            norm=cm/np.maximum(cm.sum(1,keepdims=True),1)\n            fig,ax=plt.subplots(figsize=(11,8)); im=ax.imshow(norm,vmin=0,vmax=1,cmap=\'Blues\')\n            ax.set_xticks(range(len(cols)),cols,rotation=70); ax.set_yticks(range(len(names)),names)\n            ax.set_xlabel(\'Predicted\'); ax.set_ylabel(\'True\'); ax.set_title(f\'{source} {typ}: row-normalized confusion\')\n            fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(figures/f\'{source}_{typ}_confusion.png\',dpi=160); plt.show(); plt.close(fig)\n    # Participant, posture, common-vs-expanded evidence remain visible.\n    pp=[]\n    for (pid,part,posture,typ),g in d.groupby([\'pid\',\'part\',\'posture\',\'type\'],dropna=False):\n        pp.append(dict(pid=pid,part=part,posture=posture,type=typ,n=len(g),macro_f1=macro(g,ACTIONS if typ==\'activity\' else SIGNS)))\n    pd.DataFrame(pp).to_csv(out/\'posture_participant_metrics.csv\',index=False)\n    summary=m.groupby([\'split\',\'type\']).agg(mean_f1=(\'macro_f1\',\'mean\'),sd_f1=(\'macro_f1\',\'std\'),participants=(\'pid\',\'nunique\'),events=(\'n\',\'sum\')).reset_index()\n    summary.to_csv(out/\'summary.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(9,5))\n    iso=m[m.split.eq(\'isolated\')].pivot(index=\'pid\',columns=\'type\',values=\'macro_f1\'); iso.plot.bar(ax=ax,ylim=(0,1))\n    ax.set_title(\'Unseen-participant performance\'); ax.set_ylabel(\'Macro F1\'); fig.tight_layout(); fig.savefig(figures/\'participant_f1.png\',dpi=160); plt.show(); plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9,5)); ev.set_index(\'pid\')[[\'detection_recall\',\'identity_recall\',\'intent_e2e\']].plot.bar(ax=ax,ylim=(0,1))\n    ax.set_title(\'Stories: detection, correct sign, correct intent\'); fig.tight_layout(); fig.savefig(figures/\'story_end_to_end.png\',dpi=160); plt.show(); plt.close(fig)\n    print(summary.to_string(index=False)); print(ev[[\'pid\',\'n_true\',\'detection_f1\',\'identity_f1\',\'intent_e2e\',\'false_commands_per_routine_hour\']].to_string(index=False))\n    print(\'Completed participants:\',sorted(d.pid.unique()),\'of\',PIDS)\n    if len(frames)<5: print(\'PARTIAL EVALUATION — finish the remaining folds before reporting a five-person result.\')\n    return summary\n\n\ndef load_deployment(folder,device=\'cuda\'):\n    folder=Path(folder); card=json.loads((folder/\'model_card.json\').read_text()); nets=[]; scale=None\n    for path in sorted(folder.glob(\'seed_*/best.pt\')):\n        ck=trusted_load(path); net=CSIModel(ck[\'arch\']).to(device); net.load_state_dict(ck[\'model\']); net.eval()\n        nets.append(net); scale=ck[\'scale\']\n    assert nets,\'No saved model weights\'\n    return nets,scale,card\n\n\ndef predict_unlabelled_csv(csv_path,model_folder,device=\'cuda\'):\n    nets,scale,card=load_deployment(model_folder,device)\n    s=raw_to_stream(csv_path); s.update(file=Path(csv_path).name,pid=\'new\',part=\'inference\',events=[])\n    pred=average_predictions([predict(n,[s],scale,card[\'arch\'],device=device) for n in nets])[0]\n    # Unlabelled continuous inference uses CNNs, not oracle-boundary ExtraTrees fusion.\n    rows=[]\n    for c,ps,pf,valid in zip(pred[\'centers\'],pred[\'ps\'],pred[\'pf\'],pred[\'valid\']):\n        if not valid: label=state=\'unavailable\'; confidence=0.\n        else:\n            state=STATES[int(ps.argmax())]\n            labels=ACTIONS if state==\'activity\' else SIGNS\n            f=pf[:len(ACTIONS)] if state==\'activity\' else pf[len(ACTIONS):]\n            label=\'idle\' if state==\'idle\' else labels[int(f.argmax())]\n            confidence=float(ps.max())\n        rows.append(dict(timestamp_us=int(s[\'t\'][c]),state=state,label=label,state_confidence=confidence))\n    return pd.DataFrame(rows),pd.DataFrame(commands(pred,card[\'tau\']))\n\n# Fast path: no DataLoader workers, no DataParallel. Each process owns one GPU.\ndef gpu_bank(streams,scale):\n    offsets=np.cumsum([0]+[len(s[\'x\']) for s in streams[:-1]])\n    flat=np.concatenate([np.clip(s[\'x\']/scale,-20,20) for s in streams],axis=0).T.copy()\n    bank=torch.from_numpy(flat).cuda()\n    rows=examples(streams)\n    centers=rows[:,1]+offsets[rows[:,0]]\n    return bank,torch.as_tensor(centers,device=\'cuda\'),torch.as_tensor(rows[:,2:],device=\'cuda\'),sampling_weights(streams,rows).float().cuda()\n\n\ndef fit_fast(tr,va,arch,cfg,folder,fingerprint,epochs=None,deadline=None):\n    global CURRENT_JOB\n    require_kaggle_gpu(); torch.set_num_threads(2); seed_all(cfg[\'seed\'])\n    torch.backends.cudnn.benchmark=True; torch.set_float32_matmul_precision(\'high\')\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    CURRENT_JOB=str(folder.relative_to(folder.parents[3])) if len(folder.parents)>3 else str(folder)\n    progress(\'building GPU sample bank\')\n    scale=fit_scale(tr);bank,centers,targets,weights=gpu_bank(tr,scale)\n    net=CSIModel(arch).cuda()\n    optimizer=torch.optim.AdamW(net.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    total=int(epochs or cfg[\'epochs\'])\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=total,eta_min=1e-5)\n    scaler=torch.amp.GradScaler(\'cuda\'); width=int(arch[\'seconds\']*FS)\n    times=torch.arange(width,device=\'cuda\')-width//2\n    best=-1.;best_epoch=1;best_weights=None;bad=0;history=[]\n    print(f\'[JOB] {folder}: {total} epochs maximum, {cfg["steps_per_epoch"]} batches/epoch\',flush=True)\n    for epoch in range(total):\n        if deadline is not None and time.time()>=deadline:\n            if best_weights is None: raise SessionBudgetReached(\'No epoch completed before time budget.\')\n            break\n        started=time.time();net.train();losses=[];progress(f\'epoch {epoch+1} training\')\n        picks=torch.multinomial(weights,cfg[\'steps_per_epoch\']*cfg[\'batch_size\'],replacement=True).reshape(cfg[\'steps_per_epoch\'],cfg[\'batch_size\'])\n        for step,ix in enumerate(picks):\n            if step%8==0:\n                progress(f\'epoch {epoch+1}, batch {step+1}/{cfg["steps_per_epoch"]}\')\n                if deadline is not None and time.time()>=deadline:\n                    raise SessionBudgetReached(\'GPU training budget reached between batches.\')\n            idx=centers[ix,None]+times[None,:]\n            x=bank[:,idx].permute(1,0,2).contiguous();ys=targets[ix,0];yf=targets[ix,1]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                st,fi=net(augment(x));loss=F.cross_entropy(st,ys,label_smoothing=0.03)\n                has=yf>=0\n                if has.any():loss=loss+F.cross_entropy(fi[has],yf[has],label_smoothing=0.05)\n            if not torch.isfinite(loss):raise RuntimeError(\'Nonfinite loss; previous checkpoint is retained.\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(net.parameters(),1.)\n            scaler.step(optimizer);scaler.update();losses.append(float(loss.detach()))\n        scheduler.step()\n        # Validate every two epochs, plus the last; no repeated full-stream check each epoch.\n        score=None\n        if va and ((epoch+1)%2==0 or epoch+1==total):\n            progress(f\'epoch {epoch+1} validation\')\n            score=validation_score(predict(net,va,scale,arch,cfg[\'batch_size\']))\n        improved=(not va) or (score is not None and score>best+1e-4)\n        if improved:\n            best=score if score is not None else 0.;best_epoch=epoch+1;bad=0\n            best_weights={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}\n        elif score is not None:bad+=1\n        history.append(dict(epoch=epoch+1,loss=float(np.mean(losses)),validation_score=score,seconds=time.time()-started))\n        payload=dict(version=VERSION,arch=arch,classes=FINE,states=STATES,scale=scale,dataset=fingerprint,\n            epoch=epoch+1,config=cfg,model={k:v.detach().cpu() for k,v in net.state_dict().items()},optimizer=optimizer.state_dict(),\n            rng_python=random.getstate(),rng_numpy=np.random.get_state(),rng_torch=torch.get_rng_state(),rng_cuda=torch.cuda.get_rng_state_all(),\n            scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),train_participants=sorted({s[\'pid\'] for s in tr}),\n            validation_participants=sorted({s[\'pid\'] for s in va}),history=history)\n        progress(f\'epoch {epoch+1} saving\')\n        checkpoint_save(payload,folder/\'last.pt\')\n        if improved:\n            checkpoint_save(dict(model=best_weights,arch=arch,classes=FINE,states=STATES,scale=scale,dataset=fingerprint,epoch=best_epoch),folder/\'best.pt\')\n        pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\n        print(f\'[EPOCH] {folder.name}/{arch["name"]} {epoch+1}/{total} loss={np.mean(losses):.4f} val={score} elapsed={time.time()-started:.1f}s\',flush=True)\n        progress(f\'epoch {epoch+1} completed\')\n        if va and epoch+1>=cfg[\'min_epochs\'] and bad>=cfg[\'patience\']:break\n    if best_weights is None:raise SessionBudgetReached(\'No validated model completed within the budget.\')\n    net.load_state_dict(best_weights)\n    del bank,centers,targets,weights,optimizer,scheduler,scaler,picks\n    torch.cuda.empty_cache()\n    return net,scale,best_epoch,history\n\n\ndef fast_fold(streams,test,cfg,out,fingerprint,deadline):\n    global CURRENT_JOB\n    require_kaggle_gpu();out=Path(out);fold=out/test;fold.mkdir(parents=True,exist_ok=True)\n    val=VALIDATION_PID[test]\n    tr=[s for s in streams if s[\'pid\'] not in [test,val]]\n    va=[s for s in streams if s[\'pid\']==val]\n    refit=[s for s in streams if s[\'pid\']!=test]\n    te=[s for s in streams if s[\'pid\']==test]\n    assert not {test,val}&{s[\'pid\'] for s in tr}\n    rows=[];choices=[]\n    for arch in CANDIDATES:\n        net,scale,ep,_=fit_fast(tr,va,arch,cfg,fold/\'selection\'/arch[\'name\'],fingerprint,deadline=deadline)\n        progress(\'scoring validation participant\')\n        pred=predict(net,va,scale,arch,cfg[\'batch_size\']);score=validation_score(pred)\n        rows.append(dict(candidate=arch[\'name\'],validation_pid=val,score=score,best_epoch=ep))\n        choices.append((score,arch,ep,pred,segment_table(pred),scale))\n        release(net);del net\n    pd.DataFrame(rows).to_csv(fold/\'validation_selection.csv\',index=False)\n    _,arch,ep,vpred,vd,scale=max(choices,key=lambda r:r[0])\n    tau,tau_rows=choose_tau([vpred]);pd.DataFrame(tau_rows).to_csv(fold/\'validation_thresholds.csv\',index=False)\n    progress(\'fitting validation action tree\')\n    tree=train_tree(tr,scale,fold/\'selection_action_tree.joblib\');tp=tree_predictions(tree,va,scale)\n    fusion=[]\n    for weight in [0.,0.5,1.]:\n        g=fuse_table(vd,tp,weight);g=g[g.type.eq(\'activity\')&g.part.ne(\'scripts\')]\n        fusion.append(dict(weight=weight,score=macro(g,ACTIONS)))\n    weight=max(fusion,key=lambda r:r[\'score\'])[\'weight\']\n    choice=dict(arch=arch,epochs=ep,tau=tau,action_tree_weight=weight,validation_pid=val,\n                training_pids=[p for p in PIDS if p!=test],test_pid=test,validation_fusion=fusion,\n                protocol=\'One fixed validation person within training people; single-seed refit. No full nested search.\')\n    atomic_json(choice,fold/\'selection.json\')\n    net,scale,_,_=fit_fast(refit,[],arch,cfg,fold/\'models\'/f\'seed_{cfg["seed"]}\',fingerprint,epochs=ep,deadline=deadline)\n    progress(\'predicting unseen test participant\')\n    pred=predict(net,te,scale,arch,cfg[\'batch_size\']);d=segment_table(pred)\n    release(net);del net\n    progress(\'fitting final action tree\')\n    tree=train_tree(refit,scale,fold/\'action_tree.joblib\')\n    d=fuse_table(d,tree_predictions(tree,te,scale),weight)\n    save_predictions(pred,fold/\'frame_predictions.npz\');report_fold(d,pred,tau,fold)\n    atomic_json(dict(test=test,version=VERSION,protocol=\'fast fixed-validation LOSO\'),fold/\'complete.json\')\n    print(\'[FOLD COMPLETE]\',test,choice,flush=True);progress(\'fold completed\')\n\n\ndef fast_final(streams,cfg,out,fingerprint,deadline):\n    require_kaggle_gpu();out=Path(out)\n    completed=[p for p in PIDS if (out/p/\'complete.json\').exists()]\n    choices=[json.loads((out/p/\'selection.json\').read_text()) for p in completed]\n    if choices:\n        names=[c[\'arch\'][\'name\'] for c in choices]\n        name=max([a[\'name\'] for a in CANDIDATES],key=lambda x:names.count(x))\n        relevant=[c for c in choices if c[\'arch\'][\'name\']==name]\n        arch=next(a for a in CANDIDATES if a[\'name\']==name)\n        epochs=max(4,int(np.median([c[\'epochs\'] for c in relevant])))\n        tau=float(np.median([c[\'tau\'] for c in relevant]))\n        weight=float(pd.Series([c[\'action_tree_weight\'] for c in choices]).mode().iloc[0])\n    else:\n        arch=CANDIDATES[0];epochs=12;tau=0.6;weight=0.\n    final=out/\'deployment_all5\';final.mkdir(exist_ok=True)\n    net,scale,_,_=fit_fast(streams,[],arch,cfg,final/f\'seed_{cfg["seed"]}\',fingerprint,epochs=epochs,deadline=deadline)\n    release(net);del net\n    progress(\'fitting all-five action tree\')\n    train_tree(streams,scale,final/\'action_tree.joblib\')\n    atomic_json(dict(arch=arch,epochs=epochs,tau=tau,action_tree_weight=weight,classes=FINE,fs=FS,\n        training_participants=PIDS,dataset=fingerprint,completed_evaluation_folds=completed,\n        evaluation_status=\'complete\' if len(completed)==5 else \'PARTIAL; do not report a five-fold mean\',\n        validation=\'All-five deployment refit has no independent accuracy score.\',\n        inference=\'Offline centered windows; continuous CNN inference; tree fusion only for isolated windows.\'),final/\'model_card.json\')\n    shutil.copy2(__file__,final/\'fyp_fast.py\');progress(\'deployment refit completed\')\n    print(\'[ALL-FIVE MODEL SAVED]\',final,flush=True)\n\n\ndef fast_worker(spec_path):\n    import joblib\n    global PROGRESS_PATH,CURRENT_JOB\n    spec=json.loads(Path(spec_path).read_text());PROGRESS_PATH=Path(spec[\'progress\']);CURRENT_JOB=spec[\'pid\']\n    torch.set_num_threads(2)\n    require_kaggle_gpu();progress(\'loading preprocessed data\')\n    streams=joblib.load(spec[\'streams\'])\n    try:\n        if spec[\'pid\']==\'FINAL\':fast_final(streams,spec[\'cfg\'],spec[\'out\'],spec[\'fingerprint\'],spec[\'deadline\'])\n        else:fast_fold(streams,spec[\'pid\'],spec[\'cfg\'],spec[\'out\'],spec[\'fingerprint\'],spec[\'deadline\'])\n    except SessionBudgetReached as exc:\n        print(\'[TIME BUDGET]\',exc,flush=True);progress(\'time budget reached\');return 75\n    return 0\n\n\nif __name__==\'__main__\':\n    import sys\n    if len(sys.argv)==3 and sys.argv[1]==\'--worker\':sys.exit(fast_worker(sys.argv[2]))\n', 'fast_runner.py': '"""Process supervision for bounded Kaggle runs. This module never trains on import."""\nfrom pathlib import Path\nimport os,sys,json,time,signal,subprocess,collections\n\n\ndef stop(proc):\n    if proc.poll() is not None:return\n    os.killpg(proc.pid,signal.SIGTERM)\n    try:proc.wait(timeout=5)\n    except subprocess.TimeoutExpired:\n        os.killpg(proc.pid,signal.SIGKILL);proc.wait(timeout=5)\n\n\ndef launch(spec,script):\n    path=Path(spec[\'spec_path\']);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(spec,indent=2));log=path.with_suffix(\'.log\')\n    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(spec[\'gpu\']),PYTHONUNBUFFERED=\'1\',\n             OMP_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\')\n    file=open(log,\'w\')\n    proc=subprocess.Popen([sys.executable,\'-u\',str(script),\'--worker\',str(path)],stdout=file,\n        stderr=subprocess.STDOUT,env=env,start_new_session=True)\n    return dict(spec=spec,proc=proc,file=file,log=log,pos=0,started=time.time(),notice=time.time())\n\n\ndef drain(job):\n    with open(job[\'log\']) as f:\n        f.seek(job[\'pos\']);s=f.read();job[\'pos\']=f.tell()\n    if s:\n        for line in s.splitlines():print(f\'[{job["spec"]["pid"]}] {line}\',flush=True)\n\n\ndef run_jobs(specs,script,max_parallel=2,stall_seconds=300,poll_seconds=2):\n    pending=collections.deque(specs);active=[];results=[];free=list(range(max_parallel))\n    try:\n        while pending or active:\n            while pending and free:\n                spec=dict(pending.popleft())\n                if time.time()>=spec[\'deadline\']:\n                    results.append(dict(pid=spec[\'pid\'],status=\'not_started_time_budget\'));continue\n                gpu=free.pop(0);spec[\'gpu\']=gpu\n                if spec.get(\'max_job_seconds\'):\n                    spec[\'deadline\']=min(spec[\'deadline\'],time.time()+spec[\'max_job_seconds\'])\n                print(\'[START]\',spec[\'pid\'],\'GPU\',gpu,flush=True)\n                active.append(launch(spec,script))\n            for job in list(active):\n                drain(job);spec=job[\'spec\'];now=time.time();rc=job[\'proc\'].poll();reason=None\n                last=job[\'started\'];phase=\'worker startup\'\n                progress=Path(spec[\'progress\'])\n                if progress.exists():\n                    try:\n                        data=json.loads(progress.read_text());last=data[\'time\'];phase=data[\'phase\']\n                    except (OSError,ValueError,KeyError):pass\n                if rc is None and now>=spec[\'deadline\']:\n                    reason=\'time_budget_reached\';stop(job[\'proc\']);rc=job[\'proc\'].returncode\n                elif rc is None and now-last>=stall_seconds:\n                    reason=\'stalled_no_progress\';stop(job[\'proc\']);rc=job[\'proc\'].returncode\n                if rc is not None:\n                    drain(job);job[\'file\'].close();active.remove(job);free.append(spec[\'gpu\']);free.sort()\n                    result=dict(pid=spec[\'pid\'],status=reason or (\'complete\' if rc==0 else \'failed_or_paused\'),\n                        exit_code=rc,seconds=round(now-job[\'started\'],1),last_phase=phase,log=str(job[\'log\']))\n                    results.append(result)\n                    Path(spec[\'spec_path\']).with_suffix(\'.status.json\').write_text(json.dumps(result,indent=2))\n                    print(\'[END]\',result,flush=True)\n                elif now-job[\'notice\']>=60:\n                    print(\'[STATUS]\',spec[\'pid\'],phase,\'seconds since progress:\',int(now-last),flush=True)\n                    job[\'notice\']=now\n            if pending or active:time.sleep(poll_seconds)\n    finally:\n        for job in active:\n            stop(job[\'proc\']);drain(job);job[\'file\'].close()\n    return results\n', 'fast_prepare.py': "from pathlib import Path\nimport sys,json,time,joblib\nimport fyp_fast as fyp\nspec=json.loads(Path(sys.argv[2]).read_text());cfg=spec['data'];work=Path(spec['work']);work.mkdir(parents=True,exist_ok=True)\nfyp.atomic_json(dict(time=time.time(),phase='validating and preprocessing real recordings'),spec['progress'])\nif cfg.get('root'):\n    root=Path(cfg['root'])\nelse:\n    candidates=[p.parent for p in Path('/kaggle/input').rglob('manifest.json') if (p.parent/'session_log.csv').is_file()]\n    if len(candidates)==1 and not cfg.get('zip'):\n        root=candidates[0]\n    else:\n        archives=[Path(cfg['zip'])] if cfg.get('zip') else list(Path('/kaggle/input').rglob('FYP_Phase1_P01_P05_verified.zip'))\n        assert len(archives)==1,'Set DATA_ROOT or ZIP_PATH to the attached private dataset.'\n        root=fyp.extract_verified(archives[0],Path('/kaggle/temp/fyp_fast_data'),cfg['sha'])\nstreams,fingerprint=fyp.prepare(root,work)\ncache=work/'cache';cache.mkdir(exist_ok=True)\njoblib.dump(streams,cache/'prepared_streams.joblib',compress=0)\nfyp.atomic_json(dict(fingerprint=fingerprint,streams=len(streams),participants=sorted({s['pid'] for s in streams}),dataset_root=str(root)),work/'prepared.json')\nfyp.atomic_json(dict(time=time.time(),phase='preprocessing complete'),spec['progress'])\nprint('[DATA READY]',fingerprint,len(streams),'recordings',flush=True)\n", 'fast_report.py': "from pathlib import Path\nimport json,sys,os,shutil\nos.environ['MPLBACKEND']='Agg'\nimport fyp_fast as fyp\nspec=json.loads(Path(sys.argv[2]).read_text())\nout=Path(spec['out']);out.mkdir(parents=True,exist_ok=True)\nfyp.summarize(out)\nimport matplotlib.pyplot as plt\nplt.close('all')\ncompleted=[p for p in fyp.PIDS if (out/p/'complete.json').exists()]\nfyp.atomic_json(dict(completed_test_participants=completed,evaluation_complete=len(completed)==5,\n    all_five_model_ready=(out/'deployment_all5/model_card.json').exists(),report_generated=True),out/'run_completion.json')\narchive=fyp.archive_outputs(out)\nprint('[RESULTS ZIP]',archive,flush=True)\n"}
for filename,source in SOURCES.items():(SOURCE_DIR/filename).write_text(source)
sys.path.insert(0,str(SOURCE_DIR))
import fast_runner
print("Sources written:",SOURCE_DIR)


## Validate and prepare all five participants
ZIP CRC and SHA256 checks detect incomplete uploads. The manifest checks individual files. Missing P2 action intervals are excluded; P4/P5 seated/cooking additions and packet-gap masks are included. Normalization is fitted using training participants only. Empty-room files remain quality references.

The model input is 52 subcarriers at 50 Hz. Short and severe gaps are handled with the same fixed quality rules as the previous notebook. Video performance and cue/keypress label accuracy remain unverified.

In [ ]:
prep=dict(pid="PREPARE",spec_path=str(WORK/"prepare.json"),progress=str(WORK/"prepare.progress.json"),
          deadline=min(RUN_STARTED+10*60,TRAINING_DEADLINE),work=str(WORK),
          data=dict(root=DATA_ROOT,zip=ZIP_PATH,sha=EXPECTED_ZIP_SHA256))
prep_status=fast_runner.run_jobs([prep],SOURCE_DIR/"fast_prepare.py",max_parallel=1,stall_seconds=STALL_SECONDS)
assert prep_status[0]["status"]=="complete","Dataset preparation failed or exceeded its budget; read prepare.log. No training started."
prepared=json.loads((WORK/"prepared.json").read_text())
assert prepared["participants"]==["P01","P02","P03","P04","P05"] and prepared["streams"]==22
import pandas as pd
from IPython.display import display,FileLink,Image
display(pd.read_csv(WORK/"recording_audit.csv"))
coverage=pd.read_csv(WORK/"event_coverage.csv")
display(coverage.groupby(["pid","part","covered"]).size().rename("events").reset_index())
for filename in ["recording_audit.csv","event_coverage.csv","dataset_info.json"]:shutil.copy2(WORK/filename,OUT/filename)
for filename in SOURCES:shutil.copy2(SOURCE_DIR/filename,OUT/filename)
versions={m:importlib.import_module(m).__version__ for m in ["torch","numpy","pandas","scipy","sklearn"]}
(OUT/"environment.json").write_text(json.dumps(versions,indent=2))


## Run five held-out participant tests
Candidates: the friend's adapted residual CNN at four seconds, and a compact four-second multiscale CNN. Both predict idle/activity/sign and all 26 fine classes. The candidate, stopping epoch, story threshold, and isolated-action ExtraTrees fusion weight use the fixed validation person only.

| Test person | Validation person | Initial training people |
|---|---|---|
| P1 | P4 | P2, P3, P5 |
| P2 | P5 | P1, P3, P4 |
| P3 | P1 | P2, P4, P5 |
| P4 | P2 | P1, P3, P5 |
| P5 | P3 | P1, P2, P4 |

Each selected CNN is then refitted on all four training people before scoring the held-out person. Each fit has at most 28 epochs and 32 batches per epoch; validation runs every two epochs. A single seed and fixed validation split save time. The two GPUs run different test folds; models never exchange data or predictions during training.

In [ ]:
cfg=dict(epochs=28,patience=4,min_epochs=8,steps_per_epoch=32,batch_size=256,
         lr=0.001,weight_decay=0.01,seed=17,tree_estimators=300)
(OUT/"training_config.json").write_text(json.dumps(cfg,indent=2))
common=dict(out=str(OUT),streams=str(WORK/"cache/prepared_streams.joblib"),fingerprint=prepared["fingerprint"],cfg=cfg)
specs=[]
for participant in ["P01","P02","P03","P04","P05"]:
    specs.append(dict(common,pid=participant,spec_path=str(WORK/(participant+".json")),
        progress=str(WORK/(participant+".progress.json")),deadline=EVALUATION_DEADLINE,max_job_seconds=MAX_FOLD_MINUTES*60))
evaluation_status=fast_runner.run_jobs(specs,SOURCE_DIR/"fyp_fast.py",max_parallel=GPUS,stall_seconds=STALL_SECONDS)
(OUT/"evaluation_status.json").write_text(json.dumps(evaluation_status,indent=2))
display(pd.DataFrame(evaluation_status))


## Train and save the all-five model
The final model trains on **P1+P2+P3+P4+P5**. Architecture is chosen by validation-selection votes, never by outer test performance. Training duration and threshold come from those validation decisions.

This deployment refit is saved separately and has no new independent accuracy score. If evaluation was incomplete, the model card says so. If no fold completed, the code uses a predeclared residual model and threshold rather than inventing validation evidence. A failed or timed-out final fit is reported as missing.

In [ ]:
final_spec=dict(common,pid="FINAL",spec_path=str(WORK/"FINAL.json"),progress=str(WORK/"FINAL.progress.json"),
                deadline=TRAINING_DEADLINE,max_job_seconds=MAX_FINAL_MINUTES*60)
final_status=fast_runner.run_jobs([final_spec],SOURCE_DIR/"fyp_fast.py",max_parallel=1,stall_seconds=STALL_SECONDS)
(OUT/"final_status.json").write_text(json.dumps(final_status,indent=2))
completed=[p for p in ["P01","P02","P03","P04","P05"] if (OUT/p/"complete.json").exists()]
card=OUT/"deployment_all5/model_card.json"
print("Completed participant tests:",completed)
print("All-five final model ready:",card.exists())
if card.exists():display(json.loads(card.read_text()))


## Results, graphs and downloadable model
Per-class F1 and confusion matrices are generated from real held-out predictions. Story detection, correct sign identity, full intent success, false commands per hour and posture breakdowns remain separate from oracle-boundary scores.

Only P4/P5 have isolated seated and cooking additions. Those results have two-person coverage. The new faster protocol differs from the previous full nested/multi-seed search, so compare its results with that limitation stated.

Weights and full training state are written atomically each epoch, with a previous checkpoint copy. This notebook starts fresh and does not resume old training. Save the completed Kaggle version's outputs and download the result ZIP for a separate backup.

In [ ]:
report_spec=dict(common,pid="REPORT",spec_path=str(WORK/"REPORT.json"),progress=str(WORK/"REPORT.progress.json"),
                 deadline=min(FINISH_DEADLINE,time.time()+5*60))
report_status=fast_runner.run_jobs([report_spec],SOURCE_DIR/"fast_report.py",max_parallel=1,stall_seconds=240)
status=dict(completed_test_participants=completed,evaluation_complete=len(completed)==5,
            all_five_model_ready=card.exists(),elapsed_minutes=round((time.time()-RUN_STARTED)/60,2),
            runtime_measured_on_this_run=True,report_status=report_status)
(OUT/"run_completion.json").write_text(json.dumps(status,indent=2))
display(status)
archive=OUT.parent/"FYP_all_checkpoints_and_results.zip"
if archive.exists():display(FileLink(str(archive)))
else:print("ZIP export did not finish. Individual model and result files remain under:",OUT)
for figure in sorted((OUT/"figures").glob("*.png")):display(Image(filename=str(figure)))
print("Save/download the completed version's Output files. Output directory:",OUT)


## Interpretation and verification
Neural training uses amplitude-only CSI, mild augmentation, balanced state/class/participant sampling, AdamW, gradient clipping and mixed precision. The model uses centered windows and is an offline prototype, not zero-latency live recognition. Continuous inference uses CNN predictions; ExtraTrees fusion uses known isolated windows.

Before delivery, the notebook is checked for syntax, embedded-source consistency, split disjointness, GPU-bank indexing equivalence using synthetic CPU tensors, result export, and watchdog termination of fake stalled processes. **No training is executed on the laptop.** CUDA performance and accuracy require this Kaggle run.

[PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) · [Kaggle notebooks](https://www.kaggle.com/docs/notebooks) · [InceptionTime inspiration](https://arxiv.org/abs/1909.04939)